# 03 - Precipitation Ensemble Forecasts

## Learning goals of this module
- Learn how to do a basic analysis, comparing observed precipitation with forecast precipitation from GEPS, GEFS and IFS.
- Learn about ensemble forecast verification.
- Learn how to calculate and visualize simple error metrics, such as the __rank_histogram__ and the __continuous_ranked_probability_score__.

## Assumptions
- We assume you are familiar with the concept of a __probabilistic__ forecast. 

### Reference to ensemble forecast products

- [Global Ensemble Prediction System (GEPS)](https://open.canada.ca/data/en/dataset/6d9dd2f8-202e-58cb-a110-e2168832aacb)

- [Global Ensemble Forecast System (GEFS)](https://www.ncei.noaa.gov/products/weather-climate-models/global-ensemble-forecast)

- [Integrated Forecasting System (IFS)](https://www.ecmwf.int/en/forecasts/documentation-and-support/changes-ecmwf-model)


## Run imports and set-up logging

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

from dotenv import load_dotenv

from veriflow import run_pipeline
from veriflow.constants import VERSION

# add project root (parent of notebook folder) to path
sys.path.append(str(Path("..").resolve()))


from tree_plots import (
    crps_plot,
    forecast_timeseries_plot,
    get_pair_dataset,
    rank_histogram_plot,
    scatter_plot,
)

# Reload automatically
%load_ext autoreload
%autoreload 2


warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

load_dotenv(dotenv_path="tutorial.env", override=True)

base_config = Path("config")
base_config.exists()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
logging.info(f"Running Veriflow version {VERSION}")

2026-06-18 22:48:33,809 - INFO - Running Veriflow version 0.1.0


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Inspecting the _veriflow_ pipeline configuration
1. Open the config file in the "config" directory. The name of the file is identical to the name of the notebook.
2. Inspect each of the sections to gain an understanding of what this configuration is about.

## Running the _veriflow_ pipeline

In [13]:
ods = run_pipeline((base_config / "03_elbow_precipitation_ensemble.yaml", "yaml"))

2026-06-18 22:48:33,990 - INFO - Successfully initialized the configuration. 
	 verification_period_start = 2025-05-15 00:00:00 
	 verification_period_end = 2026-06-01 00:00:00
2026-06-18 22:48:33,992 - INFO - Start getting data from FewsWebservice.
2026-06-18 22:48:34,409 - INFO - Successfully got observed data from FewsWebservice.
2026-06-18 22:48:34,410 - INFO - Start getting data from FewsWebservice.
2026-06-18 22:48:34,916 - INFO - Successfully got simulated_GEPS data from FewsWebservice.
2026-06-18 22:48:34,917 - INFO - Start getting data from FewsWebservice.
2026-06-18 22:48:35,457 - INFO - Successfully got simulated_GEFS data from FewsWebservice.
2026-06-18 22:48:35,458 - INFO - Start getting data from FewsWebservice.
2026-06-18 22:48:36,162 - INFO - Successfully got simulated_IFS data from FewsWebservice.
2026-06-18 22:48:36,166 - INFO - Successfully loaded all data from sources.
2026-06-18 22:48:36,192 - INFO - Successfully computed CrpsForEnsemble for verification pair PC_GE

## Evaluating the results in the _veriflow_ output `DataTree`

Verification metrics and results can contain a level of abstraction. Although these abstractions can reveal important information about forecast quality, a basic "eyeball verification" is often the best and intuitive way to start your verification exercise. You'll likely find strengths and weaknesses in your forecasts early on, without directly diving into levels of abstraction. In addition, a solid visual inspection may help you later on in understanding or explaining the more abstract results.

### 1 - Visual inspection of observed and forecast data
A good starting point for "eyeball" verification is simple: just looking at your observations and forecasts in a visual way. Use the interactive elements in the plots below to zoom, pan and compare the results of our 3 NWP products.

In [ ]:
stations = get_pair_dataset(ods, ods.veriflow.verification_pairs[0]).coords["station"].values
lead_times = get_pair_dataset(ods, ods.veriflow.verification_pairs[0]).coords["lead_time"].values
lead_times_hours = [lt.astype("timedelta64[h]").astype(int) for lt in lead_times]

print(f"Stations: {stations}")
print(f"Lead times (hours): {lead_times_hours}")

Stations: ['3031092']
Lead times (hours): [np.int64(24), np.int64(48), np.int64(72), np.int64(96), np.int64(120), np.int64(144)]


In [19]:
forecast_timeseries_plot(ods, show_members=False)

### 2 - Visual inspection with a scatter plot per lead time
Another great tool for "eyeball" verification is the scatter plot. The scatter plot is relatively easy to understand, but is slightly more abstract than the visualization above. 

For all forecasts in our output `DataTree`, we collect all realizations of the ensemble at a specific lead time. You can use the  `lead_times` variable  (a list of `np.timedelta64` instances) to slice the data at a specific lead time. For that given slice, we can make the scatter plot.


In [16]:
scatter_plot(ods, lead_time=lead_times[1])

### 3 - Looking into the Continuous Ranked Probability Score

Next, we'll look into the results of the continuous ranked probability score. Before you continue to the next section, we'll look into the documentation and do a tutorial on the CRPS for ensemble forecasts. The _veriflow_ package relies on _scores_ (developed by the Bureau of Meteorology, Australia) for computation of various scores. 

1. Read the [API documentation](https://scores.readthedocs.io/en/stable/api.html#scores.probability.crps_for_ensemble) the __crps_for_ensemble__ function, which is used under the hood in _veriflow_. 
2. Run through the [scores tutorial](https://scores.readthedocs.io/en/stable/tutorials/CRPS_for_Ensembles.html) on the __crps_for_ensemble__ function. You can run it in Binder (link on top of the page), or view the static view.


- Q1: what attribute(s) of forecast quality can be measured by the CRPS?

<details>
<summary>Show suggested answers for Q1</summary>
The CRPS captures multiple attributes: accuracy, reliability and sharpness. Can you reason why?

</details>

- Q2: what is the best possible CRPS score?

<details>
<summary>Show suggested answers for Q2</summary>
The best possible outcome of CRPS is 0. In this case the forecast has maximum sharpness: all ensemble members exactly predict the observed outcome.
</details>


- Q3: computing the CRPS over just one realization (i.e. a deterministic forecast) yields the exact same result as computing the ... for a deterministic score?

<details>
<summary>Show suggested answers for Q3</summary>
The absolute error. The CRPS is a probabilistic generalization of the absolute error. When taking the mean of the CRPS over all forecasts, it is equal to the mean absolute error when the number of realizations is 1.
</details>

In [17]:
crps_plot(ods)

### 4 - Looking into the Rank Histogram

Next, we'll look into the results of the rank histogram. Before you continue to the next section, we'll look into the documentation and do a tutorial on the rank histogram for ensemble forecasts. The _veriflow_ package relies on _scores_ (developed by the Bureau of Meteorology, Australia) for computation of various scores. 

1. Read the [API documentation](https://scores.readthedocs.io/en/stable/api.html#scores.probability.rank_histogram) the __rank_histogram__ function, which is used under the hood in _veriflow_. 
2. Run through the [scores tutorial](https://scores.readthedocs.io/en/stable/tutorials/Rank_Histogram.html) on the __rank_histogram__ function. You can run it in Binder (link on top of the page), or view the static view.

- Q1: what attribute(s) of forecast quality can be measured by the Rank Histogram?

<details>
<summary>Show suggested answers for Q1</summary>
The rank histogram primarily measures reliability, although bias in forecasts will show up in the rank histogram as well.
</details>

- Q2: what does a perfect Rank Histogram look like?

<details>
<summary>Show suggested answers for Q2</summary>
The rank histogram shows a uniform (flat) distribution, indicating forecast and observed frequencies are equal.
</details>

- Q3: does a perfectly uniform/flat rank histogram always correspond to a good forecast? Can you come up with a hypothetical forecast that has a perfect rank histogram, but is still bad?

<details>
<summary>Show suggested answers for Q3</summary>
No, although a perfect rank histogram indicates statistical reliability, the forecast may still have poor resolution (the ability to discriminate between situations that have different observed outcomes).

Imagine an ensemble forecast that ignores current conditions and simply samples from the historical distribution of streamflow for that day of year.

For example, every day in June the ensemble consists of random draws from the last 30 years of June observations.

This forecast can also produce a nearly uniform rank histogram because the observations come from the same climatological distribution. However, it has zero ability to distinguish today's conditions from any other June day.

</details>


In [18]:
rank_histogram_plot(ods, lead_time=lead_times[1])